In [66]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot
import numpy as np
import os
from datetime import timedelta

pd.set_option('display.max_columns',None)

In [75]:
def get_dataset(dataset:str)->str:
    try:
        return pd.read_csv(os.path.join('..','data','raw',f'{dataset}.csv'))
    except Exception as e:
        warnings.warn(f'"{dataset}" dataset not present.')
        return None

In [67]:
def calculate_distance(record):
    lat1 = np.radians( record['customer_latitude'])
    lng1 = np.radians(record['customer_longitude'])
    lat2 = np.radians(record['seller_latitude'])
    lng2 = np.radians(record['seller_longitude'])

    # Calucate the difference.
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    
    a = (np.sin(dlat/2)**2) + np.cos(lat1) * np.cos(lat2) * np.sin(dlng/2)**2

    c =  2 * np.asin(np.sqrt(a))

    r = 6371

    return np.round((c * r),0)
    

In [68]:
data = pd.read_csv(os.path.join('..','data','processed','training_data.csv'))

In [69]:
data.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_latitude,customer_longitude,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,seller_latitude,seller_longitude
0,0f678c7e594e9fd0203bb134c9bdf778,636fc788dff5948716dfe026ba361b98,delivered,2018-01-13 20:30:18,2018-01-13 20:39:26,2018-01-16 15:16:14,2018-01-18 22:10:03,2018-02-01,1.0,credit_card,2.0,60.48,8c256ad768e0fccf0025e90026c6282e,3175,sao paulo,SP,-23.549821,-46.584579,1.0,4af84afb6aeb941678e2c3c56ee66343,5f1dc28029d2c244352a68107ec2b542,2018-01-18 20:39:26,20.9,9.34,5126.0,sao paulo,SP,-23.501948,-46.741153
1,b6f41799a58be93e516fb6992e0d4b47,58284e22c11c1cd2a52d4f25630a1ed9,delivered,2017-09-17 06:44:58,2017-09-17 06:55:06,2017-09-18 21:33:16,2017-10-16 18:13:23,2017-10-17,1.0,credit_card,10.0,144.57,98758d88bf4b8eef1372ddee45d63178,57250,campo alegre,AL,-9.784927,-36.351801,1.0,cbe952607a2215211502095c631ea179,b9c0ffa92f3ff9a5a3d45ef7213c0842,2017-09-21 06:55:06,110.0,34.57,8246.0,sao paulo,SP,-23.531065,-46.437566
2,098dd2a0dabd8878380b344370974efa,971c2e479ef8a3a4a70418a4e6100228,delivered,2017-03-14 22:06:27,2017-03-14 22:06:27,2017-03-16 11:39:33,2017-03-22 06:34:07,2017-04-17,1.0,boleto,1.0,75.44,c135309fbd128284cca4af3a41518195,7051,guarulhos,SP,-23.467754,-46.551961,2.0,f845bd3077894700e688b1944854e28a,aaed1309374718fdd995ee4c58c9dfcd,2017-03-20 22:06:27,23.2,14.52,89120.0,timbo,SC,-26.825326,-49.269771


In [70]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 85068 entries, 0 to 85067
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       85068 non-null  str    
 1   customer_id                    85068 non-null  str    
 2   order_status                   85068 non-null  str    
 3   order_purchase_timestamp       85068 non-null  str    
 4   order_approved_at              84943 non-null  str    
 5   order_delivered_carrier_date   83567 non-null  str    
 6   order_delivered_customer_date  82621 non-null  str    
 7   order_estimated_delivery_date  85068 non-null  str    
 8   payment_sequential             85065 non-null  float64
 9   payment_type                   85065 non-null  str    
 10  payment_installments           85065 non-null  float64
 11  payment_value                  85065 non-null  float64
 12  customer_unique_id             85068 non-null  str    
 1

In [71]:
# Convert columns to their appropriate datatypes.

date_col = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
            'order_delivered_customer_date','order_estimated_delivery_date','shipping_limit_date']

number_col = ['payment_sequential','payment_installments','order_item_id','seller_zip_code_prefix']

for col in date_col:
    data[col] = pd.to_datetime( data[col])
    

for col in number_col:
    data[col] = data[col].astype('Int64')



In [72]:
data = data[data['order_status'] == 'delivered']

In [78]:
df_geolocation = get_dataset('geolocation')

In [90]:
df_unique_geolocation = df_geolocation.groupby('geolocation_zip_code_prefix').agg(latitude = ('geolocation_lat','median'),longitude=('geolocation_lng','median')).reset_index()

In [92]:
data[data['customer_latitude'].isna()]['customer_zip_code_prefix'].isin(df_unique_geolocation['geolocation_zip_code_prefix']).sum()

np.int64(0)

In [106]:
data[data['seller_latitude'].isna()]['seller_zip_code_prefix'].isin(df_unique_geolocation['geolocation_zip_code_prefix']).sum()

np.int64(0)

In [109]:
# Those zip code are not present in geolocation table so we need to drop them.

data.dropna(axis=0,subset=['customer_latitude','customer_longitude','seller_latitude','seller_longitude'],inplace=True)

In [136]:
# to order_approved at null values filling , first we take the mean time of order approved from order purchased which is '10 hourse' 
# then we fill the null values using order_purchase_time + 10 hours (which is aproximate)




duration = timedelta(hours=10)
data['order_approved_at'] = data['order_approved_at'].fillna(value= (data['order_purchase_timestamp']+duration))

In [148]:
# the remaining null values count are very low so we drop them directly before droping '82221 records'

data.dropna(inplace=True)
